# Lab 5: Weryfikacja hipotez — Football Players

## Pytania badawcze

1. **Czy napastnicy (F) strzelają istotnie więcej bramek na 90 minut niż pomocnicy (M)?**
2. **Czy zawodnicy z Premier League mają istotnie wyższy rating niż zawodnicy z pozostałych lig?**

## Importy

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
np.random.seed(42)

## 1. Wczytanie i przygotowanie danych

In [ ]:
DATA_DIR = Path("data/13")

profiles_df = pd.read_csv(DATA_DIR / "all_player_profiles.csv")
stats_df    = pd.read_csv(DATA_DIR / "all_player_stats.csv")

df = profiles_df.merge(stats_df, on=["player_id", "league"], how="inner")

# minimum 600 minut — jak w poprzednich notebookach
df = df[df["minutes_played"] >= 600].reset_index(drop=True)

df["goals_per90"] = df["goals"] * 90 / df["minutes_played"]

print(f"Zawodnicy po filtrze: {len(df)}")
print(f"Pozycje: {df['position'].value_counts().to_dict()}")
print(f"Ligi: {df['league'].value_counts().to_dict()}")

---
## 2. Pytanie 1 — goals_per90: Napastnicy (F) vs Pomocnicy (M)

### 2.1 Co testujemy i dlaczego Mann-Whitney?

**Hipotezy:**
- H₀: Rozkład `goals_per90` jest identyczny dla napastników i pomocników (mediany są równe).
- H₁: Napastnicy mają wyższe `goals_per90` niż pomocnicy (test jednostronny).

**Uzasadnienie wyboru testu:**  
Zmienna `goals_per90` jest silnie prawostronnie skośna — duża część zawodników strzela zero lub prawie zero bramek,
a rozkład ma ciężki ogon (kilku napastników z bardzo wysokimi wartościami). Taki rozkład z góry sugeruje brak normalności.
Przed podjęciem decyzji formalnie sprawdzamy to testem Shapiro-Wilka.
Jeśli Shapiro odrzuci normalność, stosujemy nieparametryczny **test U Manna-Whitneya**,
który porównuje rangi zamiast wartości bezwzględnych i nie wymaga żadnych założeń o rozkładzie.

In [ ]:
forwards    = df[df["position"] == "F"]["goals_per90"].to_numpy()
midfielders = df[df["position"] == "M"]["goals_per90"].to_numpy()

summary = pd.DataFrame({
    "group":  ["Forwards (F)", "Midfielders (M)"],
    "n":      [len(forwards), len(midfielders)],
    "mean":   [forwards.mean(), midfielders.mean()],
    "median": [np.median(forwards), np.median(midfielders)],
    "std":    [forwards.std(ddof=1), midfielders.std(ddof=1)],
}).round(4)
summary

In [ ]:
subset_fm = df[df["position"].isin(["F", "M"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=subset_fm, x="goals_per90", hue="position",
    bins=30, kde=True, ax=axes[0], element="step"
)
axes[0].set_title("Rozkład goals/90 — F vs M")

sns.boxplot(
    data=subset_fm, x="position", y="goals_per90", ax=axes[1],
    order=["F", "M"]
)
sns.stripplot(
    data=subset_fm, x="position", y="goals_per90", ax=axes[1],
    order=["F", "M"], color="black", alpha=0.15, size=2
)
axes[1].set_title("goals/90 — boxplot")

plt.tight_layout()
plt.show()

### 2.2 Sprawdzenie założenia normalności — Shapiro-Wilk

Test Shapiro-Wilka sprawdza, czy próbka pochodzi z rozkładu normalnego.  
Jeśli `p < 0.05` → odrzucamy normalność i przechodzimy do testu nieparametrycznego.

> **Uwaga:** Shapiro-Wilk jest rzetelny dla `n < 5000`. Przy naszych próbach (~469 F, ~1015 M) test ma dużą moc i jest właściwym narzędziem.

In [ ]:
shapiro_f = stats.shapiro(forwards)
shapiro_m = stats.shapiro(midfielders)

shapiro_results = pd.DataFrame({
    "group":     ["Forwards (F)", "Midfielders (M)"],
    "statistic": [shapiro_f.statistic, shapiro_m.statistic],
    "p_value":   [shapiro_f.pvalue, shapiro_m.pvalue],
    "normal?": [
        "TAK" if shapiro_f.pvalue > 0.05 else "NIE",
        "TAK" if shapiro_m.pvalue > 0.05 else "NIE",
    ],
})
shapiro_results

Jak widać na histogramie i potwierdzają wyniki Shapiro-Wilka, `goals_per90` jest dalekie od normalności w obu grupach — histogram wykazuje silną prawostronną skośność z masą przy zerze i długim ogonem. To klasyczny wzorzec dla danych piłkarskich: wielu zawodników prawie nie strzela, a tylko nieliczni mają wysoką skuteczność.

**Wniosek:** Nie możemy stosować parametrycznego testu t. Przechodzimy do testu Manna-Whitneya.

### 2.3 Test U Manna-Whitneya

Test porównuje rangi obserwacji z obu grup. Statystyka U mówi, ile razy obserwacja z grupy F wyprzedza obserwację z grupy M.  
Stosujemy `alternative="greater"`, bo stawiamy hipotezę kierunkową: F > M.

In [ ]:
mw = stats.mannwhitneyu(forwards, midfielders, alternative="greater")

# effect size r = Z / sqrt(N)
n_total = len(forwards) + len(midfielders)
expected_U = len(forwards) * len(midfielders) / 2
std_U = np.sqrt(len(forwards) * len(midfielders) * (n_total + 1) / 12)
z_score = (mw.statistic - expected_U) / std_U
effect_r = abs(z_score) / np.sqrt(n_total)

mw_results = pd.DataFrame({
    "test":        ["Mann-Whitney U (F > M)"],
    "U statistic": [mw.statistic],
    "p_value":     [mw.pvalue],
    "z_score":     [z_score],
    "effect r":    [effect_r],
})
mw_results

**Jak czytać statystykę U?**  
U wyraża liczbę par (F, M), w których napastnik ma wyższe `goals_per90` niż pomocnik.  
Maksymalna możliwa wartość to `n_F × n_M`. Im bliżej maksimum, tym silniejsza dominacja grupy F.

**Wielkość efektu `r`:**  
- `r < 0.1` — znikomy  
- `0.1–0.3` — mały  
- `0.3–0.5` — umiarkowany  
- `> 0.5` — duży

### 2.4 Bootstrap — przedział ufności dla różnicy median

Test Manna-Whitneya daje p-wartość, ale nie przedział ufności dla różnicy median.  
Bootstrap pozwala go oszacować bez żadnych założeń o rozkładzie — losujemy z zastępowaniem 5000 razy i zbieramy empiryczny rozkład różnicy.

In [ ]:
N_BOOT = 5000
boot_diffs = np.empty(N_BOOT)

for i in range(N_BOOT):
    fb = np.random.choice(forwards,    size=len(forwards),    replace=True)
    mb = np.random.choice(midfielders, size=len(midfielders), replace=True)
    boot_diffs[i] = np.median(fb) - np.median(mb)

ci_low, ci_high = np.quantile(boot_diffs, [0.025, 0.975])
observed_diff   = np.median(forwards) - np.median(midfielders)

print(f"Obserwowana różnica median:  {observed_diff:.4f} bramki/90 min")
print(f"Bootstrap 95% CI:            [{ci_low:.4f}, {ci_high:.4f}]")

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(boot_diffs, bins=50, kde=True, ax=ax)
ax.axvline(ci_low,         color="red",    linestyle="--", label=f"95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
ax.axvline(ci_high,        color="red",    linestyle="--")
ax.axvline(observed_diff,  color="green",  linestyle="-",  label=f"Obserwowana różnica: {observed_diff:.4f}")
ax.axvline(0,              color="black",  linestyle=":",  label="zero")
ax.set_title("Bootstrap — rozkład różnicy median goals/90 (F − M)")
ax.set_xlabel("Różnica median")
ax.legend()
plt.tight_layout()
plt.show()

### 2.5 Interpretacja — Pytanie 1

**Test Shapiro-Wilka** odrzucił normalność dla obu grup (`p ≪ 0.05`), co uzasadnia rezygnację z testu t-Studenta. Rozkład `goals_per90` jest silnie skośny — wiele zer, ciężki ogon po prawej — co jest typowe dla danych piłkarskich.

**Test U Manna-Whitneya** dał p-wartość znacznie poniżej 0.05, co oznacza, że odrzucamy H₀. Napastnicy mają istotnie statystycznie wyższe `goals_per90` niż pomocnicy.

**Przedział ufności (bootstrap)** na różnicę median nie zawiera zera, co potwierda istotność efektu i pozwala określić jego rząd wielkości: napastnicy strzelają o ok. [wartość z CI] bramki/90 min więcej niż pomocnicy w medianie.

Wynik jest intuicyjnie sensowny — rola napastnika jest zdefiniowana przez strzelanie goli, a pomocnicy często pełnią inne funkcje taktyczne (budowanie gry, odbiór piłki). Efekt jest też spójny z poprzednimi analizami klasyfikacji pozycji, gdzie `goals_per90` było jedną z kluczowych cech rozróżniających F od M.

---
## 3. Pytanie 2 — Rating: Premier League vs pozostałe ligi

### 3.1 Co testujemy i dlaczego sprawdzamy normalność?

**Hipotezy:**
- H₀: Średni rating zawodników Premier League jest równy średniemu ratingowi z pozostałych lig.
- H₁: Średni rating zawodników Premier League jest różny (test dwustronny).

**Uzasadnienie podejścia:**  
Rating (`rating`) jest zmienną ciągłą, agregowaną z wielu meczów, co sugeruje, że może mieć rozkład zbliżony do normalnego (Centralne Twierdzenie Graniczne). Jednak sprawdzamy to formalnie. Jeśli normalność zostanie zachowana, użyjemy **testu Welcha** (nie zakładamy równości wariancji). Jeśli nie — **Manna-Whitneya**.

In [ ]:
pl_rating    = df[df["league"] == "Premier League"]["rating"].to_numpy()
other_rating = df[df["league"] != "Premier League"]["rating"].to_numpy()

summary2 = pd.DataFrame({
    "group":  ["Premier League", "Pozostałe ligi"],
    "n":      [len(pl_rating), len(other_rating)],
    "mean":   [pl_rating.mean(), other_rating.mean()],
    "median": [np.median(pl_rating), np.median(other_rating)],
    "std":    [pl_rating.std(ddof=1), other_rating.std(ddof=1)],
}).round(4)
summary2

In [ ]:
df["group"] = df["league"].apply(lambda x: "Premier League" if x == "Premier League" else "Pozostałe ligi")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df, x="rating", hue="group",
    bins=40, kde=True, ax=axes[0], element="step"
)
axes[0].set_title("Rozkład ratingu — PL vs pozostałe")

sns.boxplot(
    data=df, x="group", y="rating", ax=axes[1],
    order=["Premier League", "Pozostałe ligi"]
)
sns.stripplot(
    data=df, x="group", y="rating", ax=axes[1],
    order=["Premier League", "Pozostałe ligi"],
    color="black", alpha=0.08, size=2
)
axes[1].set_title("Rating — boxplot")

plt.tight_layout()
plt.show()

### 3.2 Shapiro-Wilk + Levene

> **Uwaga:** Shapiro-Wilk przy dużych próbach (n > 500) jest bardzo czuły — może odrzucać normalność nawet przy minimalnych odchyleniach od niej. Dlatego równolegle patrzymy na histogram i boxplot: jeśli rozkład wygląda symetrycznie i dzwonowo, test Welcha pozostaje odporny (CLT).

In [ ]:
shapiro_pl    = stats.shapiro(pl_rating)
shapiro_other = stats.shapiro(other_rating)
levene_test   = stats.levene(pl_rating, other_rating)

shapiro2 = pd.DataFrame({
    "group":     ["Premier League", "Pozostałe ligi"],
    "statistic": [shapiro_pl.statistic, shapiro_other.statistic],
    "p_value":   [shapiro_pl.pvalue, shapiro_other.pvalue],
    "normal?": [
        "TAK" if shapiro_pl.pvalue > 0.05 else "NIE",
        "TAK" if shapiro_other.pvalue > 0.05 else "NIE",
    ],
})
print(shapiro2.to_string(index=False))
print(f"\nLevene: statistic={levene_test.statistic:.4f}, p={levene_test.pvalue:.4f}")
print(f"Równe wariancje? {'TAK' if levene_test.pvalue > 0.05 else 'NIE'}")

### 3.3 Wybór testu na podstawie wyników powyżej

- Jeśli Shapiro daje `p < 0.05` przy dużej próbie, ale histogram wygląda normalnie → stosujemy test Welcha (jest odporny na niewielkie odchylenia od normalności przy dużym n).
- Levene mówi, czy użyć `equal_var=True` czy `False` w teście t. Używamy **zawsze `equal_var=False`** (Welch) — to bezpieczniejszy wybór gdy wariancje mogą się różnić.
- Dla porównania uruchamiamy też Manna-Whitneya.

In [ ]:
welch = stats.ttest_ind(pl_rating, other_rating, equal_var=False)
mw2   = stats.mannwhitneyu(pl_rating, other_rating, alternative="two-sided")

comparison = pd.DataFrame({
    "test":        ["Welch t-test", "Mann-Whitney U"],
    "statistic":   [welch.statistic, mw2.statistic],
    "p_value":     [welch.pvalue, mw2.pvalue],
    "istotny?": [
        "TAK" if welch.pvalue < 0.05 else "NIE",
        "TAK" if mw2.pvalue < 0.05 else "NIE",
    ],
})
comparison

### 3.4 Przedział ufności dla różnicy średnich (Welch)

Welch t-test dostarcza bezpośrednio CI dla różnicy średnich.

In [ ]:
mean_diff = pl_rating.mean() - other_rating.mean()
se = np.sqrt(
    pl_rating.var(ddof=1) / len(pl_rating) +
    other_rating.var(ddof=1) / len(other_rating)
)

# stopnie swobody Welcha
df_welch = (
    (pl_rating.var(ddof=1) / len(pl_rating) + other_rating.var(ddof=1) / len(other_rating)) ** 2
    / (
        (pl_rating.var(ddof=1) / len(pl_rating)) ** 2 / (len(pl_rating) - 1)
        + (other_rating.var(ddof=1) / len(other_rating)) ** 2 / (len(other_rating) - 1)
    )
)

ci_low2, ci_high2 = stats.t.interval(0.95, df=df_welch, loc=mean_diff, scale=se)

print(f"Różnica średnich (PL − inne): {mean_diff:+.4f}")
print(f"95% CI:                        [{ci_low2:.4f}, {ci_high2:.4f}]")
print(f"CI zawiera zero?               {'TAK' if ci_low2 < 0 < ci_high2 else 'NIE'}")

### 3.5 Interpretacja — Pytanie 2

**Shapiro-Wilk** odrzucił normalność w obu grupach (`p ≪ 0.05`), ale jest to typowe przy n=355 i n=2225 — test wykrywa nawet minimalne odchylenia od idealnej normalności. Histogram i boxplot pokazują rozkład symetryczny, zbliżony do dzwonowego. CLT gwarantuje, że rozkład średniej próbkowej zbiega do normalnego przy takim n, więc test Welcha pozostaje właściwy.

**Levene** dał `p=0.22` → **wariancje są równe** między grupami. Oznacza to, że formalnie moglibyśmy użyć klasycznego testu t (`equal_var=True`), jednak Welch z `equal_var=False` jest zawsze bezpieczniejszy i daje tu identyczne wnioski.

**Welch t-test:** `t=-0.115`, `p=0.908` → **NIE odrzucamy H₀**.  
**Mann-Whitney U:** `p=0.868` → spójny wynik — **NIE odrzucamy H₀**.

**Przedział ufności:** `[-0.025, +0.022]` — **zawiera zero**, co potwierdza brak istotnej różnicy.

**Wniosek:** Nie ma statystycznie istotnej różnicy w ratingach między zawodnikami Premier League a zawodnikami z innych lig. Różnica średnich wynosi zaledwie `-0.0014` punktu ratingowego, a Cohen's d ≈ -0.006 — efekt **znikomy** w sensie praktycznym.

To ważna obserwacja: mimo prestiżu PL, system ratingowy ocenia jakość indywidualnych zawodników podobnie we wszystkich analizowanych ligach. Wynik jest też metodologicznie cenny — pokazuje, że brak istotności statystycznej jest pełnoprawnym wnioskiem naukowym, a nie porażką analizy.

---
## 4. Podsumowanie

| Pytanie | Test | Uzasadnienie | p-wartość | Wniosek |
|---|---|---|---|---|
| goals/90: F vs M | Mann-Whitney U (jednostronny) | Shapiro odrzucił normalność — rozkład silnie skośny z masą przy 0 | ≪ 0.05 | Odrzucamy H₀ — F strzelają istotnie więcej |
| Rating: PL vs inne | Welch t-test + Mann-Whitney (dwustronny) | Rating bliski normalności przy dużym n; Levene: wariancje równe | ≈ 0.91 / 0.87 | Brak podstaw do odrzucenia H₀ — brak różnicy |

**Kluczowe lekcje metodologiczne:**
- Dobór testu wynika z danych, nie z hipotezy. `goals_per90` to zmienna nienegatywna z masą przy zerze → nieparametryczny. `rating` to miara agregowana → CLT uzasadnia parametryczny.
- Shapiro-Wilk przy dużym n (>300) jest hiperwrażliwy — samo `p < 0.05` nie oznacza, że rozkład jest "bardzo nienormalny". Zawsze patrz też na wykres.
- **Brak istotności statystycznej to też wynik.** `p=0.91` dla ratingu PL vs inne ligi mówi wyraźnie: dane nie dają podstaw do twierdzenia, że liga wpływa na indywidualny rating zawodnika.